# Yahoo Finance acquisition audit

Human review of source coverage and data warnings before event-study analysis. Core acquisition, return, and audit logic stays in `price_diffusion.data`. Yahoo history is not assumed to be survivorship-free, point-in-time, or complete.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from price_diffusion.data import load_security_master, run_yahoo_pipeline

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
pd.set_option('display.max_colwidth', 140)

## Acquisition control

Set `RUN_DOWNLOAD = True` only when a new immutable Yahoo snapshot is intended. Each run creates a new raw directory; existing raw files are never overwritten.

In [ ]:
RUN_DOWNLOAD = False
if RUN_DOWNLOAD:
    artifacts = run_yahoo_pipeline()
    print(f'Raw snapshot: {artifacts.raw_snapshot_dir}')

security_master = load_security_master(PROJECT_ROOT / 'metadata' / 'security_master.csv')
display(security_master)
print(f'Approved securities: {security_master.security_id.nunique()}')

## Download status

Failures and empty Yahoo responses remain visible rather than causing the security to disappear from review.

In [ ]:
status_path = PROJECT_ROOT / 'outputs' / 'diagnostics' / 'yahoo_download_status.csv'
if status_path.exists():
    download_status = pd.read_csv(status_path, keep_default_na=False)
    display(download_status.groupby('status', dropna=False).size().rename('securities').to_frame())
    display(download_status)
else:
    download_status = pd.DataFrame()
    print('No download status exists yet. Set RUN_DOWNLOAD = True to create one.')

## Coverage and warnings

In [ ]:
audit_path = PROJECT_ROOT / 'outputs' / 'diagnostics' / 'data_audit.csv'
prices_path = PROJECT_ROOT / 'data' / 'interim' / 'daily_prices.parquet'
if not audit_path.exists() or not prices_path.exists():
    raise FileNotFoundError('Run the acquisition pipeline before reviewing coverage.')

audit = pd.read_csv(audit_path, parse_dates=['first_date', 'last_date'], keep_default_na=False)
prices = pd.read_parquet(prices_path)
display(audit.sort_values(['observation_count', 'security_id']))

`missing_percentage` is a Monday–Friday screening estimate. It includes valid local exchange holidays and must not be interpreted as an exchange-calendar error rate.

In [ ]:
coverage_plot = audit.sort_values('missing_percentage', ascending=True)
fig, ax = plt.subplots(figsize=(10, max(6, 0.22 * len(coverage_plot))))
ax.barh(coverage_plot['ticker'], coverage_plot['missing_percentage'], color='#4C78A8')
ax.set(title='Estimated missing weekdays or holidays by security', xlabel='Percent of Monday–Friday dates', ylabel='')
ax.grid(axis='x', alpha=0.25)
plt.tight_layout()

## Monthly observation map

This visualization distinguishes late listings and broad gaps from isolated missing sessions. A colored cell means at least one observation exists in that security-month.

In [ ]:
observed = prices[['ticker', 'date']].copy()
observed['month'] = observed['date'].dt.to_period('M').dt.to_timestamp()
observed['present'] = 1
monthly = observed.pivot_table(index='ticker', columns='month', values='present', aggfunc='max')
monthly = monthly.reindex(sorted(monthly.index))
fig, ax = plt.subplots(figsize=(16, max(7, 0.22 * len(monthly))))
image = ax.imshow(monthly.notna().to_numpy(), aspect='auto', interpolation='nearest', cmap='Blues')
tick_positions = np.arange(0, len(monthly.columns), 12)
ax.set_xticks(tick_positions, [monthly.columns[i].strftime('%Y') for i in tick_positions], rotation=45, ha='right')
ax.set_yticks(np.arange(len(monthly.index)), monthly.index)
ax.set(title='Months with at least one Yahoo observation', xlabel='Month', ylabel='Security')
plt.tight_layout()

## Problematic securities and review checklist

In [ ]:
problematic = audit.loc[audit['issues'].ne('') | audit['observation_count'].lt(252)].sort_values(['observation_count', 'ticker'])
display(problematic)
print(f'Securities requiring review: {len(problematic)} of {len(audit)}')

Before empirical use, review: zero-coverage and short-history listings; large gaps around ticker/listing changes; extreme returns against corporate-action records; zero-volume clusters; ADR versus primary-listing choices; exchange calendars and timezone labels; and whether local-currency returns match the intended investor perspective. No currency conversion is performed in Stage 11C.